# EDA Benchmark Suite: Minimax Search Agents (v15 – v17)

Dedicated analytical suite to audit 1-ply Minimax evaluation functions, horizon effects, search latency, and hybrid safety overrides.

In [ ]:
# Cell 1: Global Configuration & Target Agent Selection
import os
import glob
import json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── TARGET AGENT SELECTION ───────────────────────────────────────────────────
TARGET_AGENT = 'v15'

ROOT_DIR = os.path.abspath(os.getcwd())
curr = ROOT_DIR
while curr != '/' and not os.path.exists(os.path.join(curr, 'data/benchmarks/all_10k/gen9randombattle')):
    curr = os.path.dirname(curr)
if os.path.exists(os.path.join(curr, 'data/benchmarks/all_10k/gen9randombattle')):
    ROOT_DIR = curr

BENCHMARK_DIR = os.path.join(ROOT_DIR, 'data/benchmarks/all_10k/gen9randombattle')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'src/p00_core/reporting/agents', TARGET_AGENT)
os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['savefig.dpi'] = 300

print(f"🎯 Target Agent ('us'): {TARGET_AGENT}")
print(f"📁 Benchmark Directory: {BENCHMARK_DIR}")
print(f"💾 Export Directory: {OUTPUT_DIR}")

In [ ]:
# Cell 2: Data Loading & Preprocessing
files_us = sorted(glob.glob(os.path.join(BENCHMARK_DIR, f"{TARGET_AGENT}_vs_*.csv")))
print(f"Searching in: {BENCHMARK_DIR}")
print(f"Found {len(files_us)} matchup files where {TARGET_AGENT} is evaluated as 'us'")

dfs = []
for f in files_us:
    df_t = pd.read_csv(f)
    df_t['matchup_opponent'] = df_t['opponent']
    dfs.append(df_t)

if dfs:
    df_raw = pd.concat(dfs, ignore_index=True)
    print(f"✅ Total games loaded: {len(df_raw):,}")
else:
    raise FileNotFoundError(f"No benchmark CSV files found for target agent: '{TARGET_AGENT}'")

df = df_raw.copy()
df['won_bool'] = df['won'].astype(bool)
df['hp_diff'] = df['remaining_pokemon_us'] - df['remaining_pokemon_opp']
df['fainted_diff'] = df['fainted_opp'] - df['fainted_us']
df['total_switches_us'] = df['voluntary_switches_us'] + df['forced_switches_us']
df['total_switches_opp'] = df['voluntary_switches_opp'] + df['forced_switches_opp']
df['switch_diff'] = df['total_switches_us'] - df['total_switches_opp']
df['vol_switch_diff'] = df['voluntary_switches_us'] - df['voluntary_switches_opp']
df['crit_diff'] = df['crit_us'] - df['crit_opp']
df['miss_diff'] = df['miss_us'] - df['miss_opp']
df['se_diff'] = df['supereffective_us'] - df['supereffective_opp']

# Safe column initialization
df['hazard_net_us'] = (df['hazard_sets_us'] - df['hazard_sets_opp']) if 'hazard_sets_us' in df.columns else 0
df['setup_diff'] = (df['setup_uses_us'] - df['setup_uses_opp']) if 'setup_uses_us' in df.columns else 0
df['ko_check_diff'] = (df['ko_checks_us'] - df['ko_checks_opp']) if 'ko_checks_us' in df.columns else 0

print("✅ Data preprocessing and feature engineering complete.")

In [ ]:
# Cell 3: Overall Win Rate Summary
wr_summary = df.groupby('matchup_opponent').agg(
    games=('won_bool', 'count'),
    wins=('won_bool', 'sum'),
    win_rate=('won_bool', lambda x: x.mean() * 100),
    avg_turns=('turns', 'mean'),
    avg_hp_us=('remaining_pokemon_us', 'mean'),
    avg_hp_opp=('remaining_pokemon_opp', 'mean')
).sort_values(by='win_rate', ascending=False)

wr_summary.to_csv(os.path.join(OUTPUT_DIR, f"{TARGET_AGENT}_win_rate_summary.csv"))
print(f"🏆 Overall Win Rate: {df['won_bool'].mean()*100:.2f}% across {len(df):,} games")
wr_summary

In [ ]:
# Cell 4: Win Rate Bar Chart across Gauntlet
plt.figure(figsize=(12, 6))
sns.barplot(x=wr_summary.index, y=wr_summary['win_rate'], palette='crest', hue=wr_summary.index, legend=False)
plt.axhline(50, color='red', linestyle='--', label='50% Win Rate Baseline')
plt.title(f"Win Rate (%) of {TARGET_AGENT} Across All Opponents", fontsize=14, fontweight='bold')
plt.ylabel("Win Rate (%)")
plt.xlabel("Opponent")
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"{TARGET_AGENT}_win_rate_bar.png"))
plt.show()

In [ ]:
# Cell 5: Turns Boxplot Distribution
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='matchup_opponent', y='turns', palette='Set3', hue='matchup_opponent', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title(f"Match Duration (Turns) per Opponent ({TARGET_AGENT})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"{TARGET_AGENT}_turns_boxplot.png"))
plt.show()

In [ ]:
# Cell 6: Switching Tactics (Voluntary vs Forced)
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='matchup_opponent', y='voluntary_switches_us', palette='Blues', hue='matchup_opponent', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title(f"Voluntary Switches per Game ({TARGET_AGENT})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"{TARGET_AGENT}_voluntary_switches.png"))
plt.show()

## Minimax Lookahead & Search Dynamics

#### Cell 7: Search Time / Decisions Latency Audit

In [ ]:
# Cell 7: Search Time / Decisions Latency Audit
plt.figure()
sns.histplot(data=df, x='decisions_us', kde=True, color='teal')
plt.title('Minimax Decisions Executed per Game')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{TARGET_AGENT}_minimax_decisions.png'))
plt.show()

#### Cell 8: Minimax Switching vs Defensive Horizon

In [ ]:
# Cell 8: Minimax Switching vs Defensive Horizon
plt.figure()
sns.boxplot(data=df, x='matchup_opponent', y='voluntary_switches_us', palette='Blues', hue='matchup_opponent', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Minimax Voluntary Switches (1-Ply Lookahead)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{TARGET_AGENT}_minimax_switches.png'))
plt.show()

#### Cell 9: Hybrid Heuristic Override Audit (v17 Hybrid)

In [ ]:
# Cell 9: Hybrid Heuristic Override Audit (v17 Hybrid)
if 'ko_checks_us' in df.columns:
    plt.figure()
    sns.barplot(data=df, x='matchup_opponent', y='ko_checks_us', hue='matchup_opponent', legend=False)
    plt.xticks(rotation=45, ha='right')
    plt.title('Hybrid Safety Override Triggers per Opponent')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{TARGET_AGENT}_hybrid_overrides.png'))
    plt.show()

#### Cell 10: Minimax Super-Effective Exploitation

In [ ]:
# Cell 10: Minimax Super-Effective Exploitation
plt.figure()
sns.scatterplot(data=df.sample(min(5000, len(df))), x='supereffective_us', y='remaining_pokemon_opp', hue='won_bool', alpha=0.4)
plt.title('Super-Effective Pressure vs Opponent Fainted Pokemon')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{TARGET_AGENT}_se_vs_fainted.png'))
plt.show()

#### Cell 11: 1-Ply Minimax Horizon Limitation Audit (Turns in Losses)

In [ ]:
# Cell 11: 1-Ply Minimax Horizon Limitation Audit (Turns in Losses)
plt.figure()
sns.violinplot(data=df, x='won_bool', y='turns', palette='Set2')
plt.title('Game Duration (Turns) in Minimax Wins vs Losses')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{TARGET_AGENT}_minimax_turns_outcome.png'))
plt.show()

## Deep Opponent Diagnostic Matrix

#### Cell 12: Matchup Diagnostic Rank #1

In [ ]:
# Cell 12: Detailed Stats for Matchup Rank #1
if len(wr_summary) >= 1:
    opp_name = wr_summary.index[0]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #1: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 13: Matchup Diagnostic Rank #2

In [ ]:
# Cell 13: Detailed Stats for Matchup Rank #2
if len(wr_summary) >= 2:
    opp_name = wr_summary.index[1]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #2: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 14: Matchup Diagnostic Rank #3

In [ ]:
# Cell 14: Detailed Stats for Matchup Rank #3
if len(wr_summary) >= 3:
    opp_name = wr_summary.index[2]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #3: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 15: Matchup Diagnostic Rank #4

In [ ]:
# Cell 15: Detailed Stats for Matchup Rank #4
if len(wr_summary) >= 4:
    opp_name = wr_summary.index[3]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #4: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 16: Matchup Diagnostic Rank #5

In [ ]:
# Cell 16: Detailed Stats for Matchup Rank #5
if len(wr_summary) >= 5:
    opp_name = wr_summary.index[4]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #5: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 17: Matchup Diagnostic Rank #6

In [ ]:
# Cell 17: Detailed Stats for Matchup Rank #6
if len(wr_summary) >= 6:
    opp_name = wr_summary.index[5]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #6: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 18: Matchup Diagnostic Rank #7

In [ ]:
# Cell 18: Detailed Stats for Matchup Rank #7
if len(wr_summary) >= 7:
    opp_name = wr_summary.index[6]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #7: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 19: Matchup Diagnostic Rank #8

In [ ]:
# Cell 19: Detailed Stats for Matchup Rank #8
if len(wr_summary) >= 8:
    opp_name = wr_summary.index[7]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #8: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 20: Matchup Diagnostic Rank #9

In [ ]:
# Cell 20: Detailed Stats for Matchup Rank #9
if len(wr_summary) >= 9:
    opp_name = wr_summary.index[8]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #9: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 21: Matchup Diagnostic Rank #10

In [ ]:
# Cell 21: Detailed Stats for Matchup Rank #10
if len(wr_summary) >= 10:
    opp_name = wr_summary.index[9]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #10: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 22: Matchup Diagnostic Rank #11

In [ ]:
# Cell 22: Detailed Stats for Matchup Rank #11
if len(wr_summary) >= 11:
    opp_name = wr_summary.index[10]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #11: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 23: Matchup Diagnostic Rank #12

In [ ]:
# Cell 23: Detailed Stats for Matchup Rank #12
if len(wr_summary) >= 12:
    opp_name = wr_summary.index[11]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #12: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 24: Matchup Diagnostic Rank #13

In [ ]:
# Cell 24: Detailed Stats for Matchup Rank #13
if len(wr_summary) >= 13:
    opp_name = wr_summary.index[12]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #13: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 25: Matchup Diagnostic Rank #14

In [ ]:
# Cell 25: Detailed Stats for Matchup Rank #14
if len(wr_summary) >= 14:
    opp_name = wr_summary.index[13]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #14: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 26: Matchup Diagnostic Rank #15

In [ ]:
# Cell 26: Detailed Stats for Matchup Rank #15
if len(wr_summary) >= 15:
    opp_name = wr_summary.index[14]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #15: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 27: Matchup Diagnostic Rank #16

In [ ]:
# Cell 27: Detailed Stats for Matchup Rank #16
if len(wr_summary) >= 16:
    opp_name = wr_summary.index[15]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #16: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 28: Matchup Diagnostic Rank #17

In [ ]:
# Cell 28: Detailed Stats for Matchup Rank #17
if len(wr_summary) >= 17:
    opp_name = wr_summary.index[16]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #17: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 29: Matchup Diagnostic Rank #18

In [ ]:
# Cell 29: Detailed Stats for Matchup Rank #18
if len(wr_summary) >= 18:
    opp_name = wr_summary.index[17]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #18: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 30: Matchup Diagnostic Rank #19

In [ ]:
# Cell 30: Detailed Stats for Matchup Rank #19
if len(wr_summary) >= 19:
    opp_name = wr_summary.index[18]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #19: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 31: Matchup Diagnostic Rank #20

In [ ]:
# Cell 31: Detailed Stats for Matchup Rank #20
if len(wr_summary) >= 20:
    opp_name = wr_summary.index[19]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #20: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 32: Matchup Diagnostic Rank #21

In [ ]:
# Cell 32: Detailed Stats for Matchup Rank #21
if len(wr_summary) >= 21:
    opp_name = wr_summary.index[20]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #21: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 33: Matchup Diagnostic Rank #22

In [ ]:
# Cell 33: Detailed Stats for Matchup Rank #22
if len(wr_summary) >= 22:
    opp_name = wr_summary.index[21]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #22: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 34: Matchup Diagnostic Rank #23

In [ ]:
# Cell 34: Detailed Stats for Matchup Rank #23
if len(wr_summary) >= 23:
    opp_name = wr_summary.index[22]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #23: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 35: Matchup Diagnostic Rank #24

In [ ]:
# Cell 35: Detailed Stats for Matchup Rank #24
if len(wr_summary) >= 24:
    opp_name = wr_summary.index[23]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #24: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 36: Matchup Diagnostic Rank #25

In [ ]:
# Cell 36: Detailed Stats for Matchup Rank #25
if len(wr_summary) >= 25:
    opp_name = wr_summary.index[24]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #25: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 37: Matchup Diagnostic Rank #26

In [ ]:
# Cell 37: Detailed Stats for Matchup Rank #26
if len(wr_summary) >= 26:
    opp_name = wr_summary.index[25]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #26: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

#### Cell 38: Matchup Diagnostic Rank #27

In [ ]:
# Cell 38: Detailed Stats for Matchup Rank #27
if len(wr_summary) >= 27:
    opp_name = wr_summary.index[26]
    df_sub = df[df['matchup_opponent'] == opp_name]
    print(f"=== MATCHUP DIAGNOSTIC #27: {TARGET_AGENT} vs {opp_name} ===")
    print(f"Win Rate: {df_sub['won_bool'].mean()*100:.2f}% ({len(df_sub):,} games)")
    print(f"Avg Turns: {df_sub['turns'].mean():.2f}")
    print(f"Avg HP Us: {df_sub['remaining_pokemon_us'].mean():.2f} | Opp: {df_sub['remaining_pokemon_opp'].mean():.2f}")
    print(f"Avg Voluntary Switches Us: {df_sub['voluntary_switches_us'].mean():.2f} | Opp: {df_sub['voluntary_switches_opp'].mean():.2f}")

## Automatic Executive Report Export

In [ ]:
# Cell 39: Generate Executive Markdown Summary Report
report_path = os.path.join(OUTPUT_DIR, f"{TARGET_AGENT}_eda_executive_report.md")
best_opp = wr_summary.index[0]
worst_opp = wr_summary.index[-1]
best_wr = wr_summary['win_rate'].iloc[0]
worst_wr = wr_summary['win_rate'].iloc[-1]

content = f"""# Executive Benchmark Analysis Report: {TARGET_AGENT}

- **Target Agent**: `{TARGET_AGENT}`
- **Total Games Evaluated**: {len(df):,}
- **Overall Win Rate**: {df['won_bool'].mean()*100:.2f}%
- **Average Match Duration**: {df['turns'].mean():.2f} turns

## Matchup Breakdown Table

{wr_summary.to_markdown()}

## Key Findings
1. Highest Win Rate Against: `{best_opp}` ({best_wr:.1f}%)
2. Hardest Opponent: `{worst_opp}` ({worst_wr:.1f}%)
3. Average Voluntary Switches per game: {df['voluntary_switches_us'].mean():.2f}
4. Total Super-Effective Hits: {df['supereffective_us'].sum():,}
"""

with open(report_path, 'w') as f:
    f.write(content)

print(f"✅ Executive EDA report generated and saved to: {report_path}")
print(f"🎉 Analysis pipeline complete! All plots saved in: {OUTPUT_DIR}")